In [ ]:
#!/usr/bin/env python3

"""
AIMES Experiment 2A
Fixed Multi-Value Steering Generation.

Runs ONLY:
    fixed_multi

Controller:

    h' = h + gamma * sum_k g_k u_k

where:
    g_k in {+1, -1}
    u_k is the unit value direction at the selected layer.

There is NO observer and NO adaptive alpha.

Expected:
    100 prompts
    x 3 objectives
    x 10 layers
    = 3000 generations/model

HF output:

artifacts/multivalue_steering/<model>/v1/
generations/fixed_multi/

    multivalue_generations.csv
    multivalue_generations.jsonl
    generation_metadata.json
"""



# 0. INSTALL


!pip install -q -U \
    transformers \
    accelerate \
    huggingface_hub \
    safetensors \
    "pandas==2.2.3"



# 1. IMPORTS


import gc
import json
import random
import hashlib

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoProcessor,
    Gemma3ForConditionalGeneration,
)

from huggingface_hub import (
    HfApi,
    hf_hub_download,
    CommitOperationAdd,
)

from safetensors.torch import load_file

from google.colab import userdata, drive



# 2. MODEL SELECTION
#
# For your current missing artifact:
#     KEEP qwen-14b


MODEL_KEY = "gemma-4b"
MODEL_KEY = "qwen-4b"
MODEL_KEY = "llama-8b"
MODEL_KEY = "gemma-12b"
MODEL_KEY = "qwen-14b"



# 3. GLOBAL CONFIG


HF_REPO_ID = ""

VERSION = "v1"

METHOD = "fixed_multi"


HF_TOKEN = userdata.get(
    "HF_TOKEN"
)


assert HF_TOKEN is not None, (
    "HF_TOKEN missing from Colab Secrets."
)


hf_api = HfApi(
    token=HF_TOKEN
)



# 4. MODEL CONFIGS


MODEL_CONFIGS = {

    "gemma-4b": {

        "model_id":
            "google/gemma-3-4b-it",

        "save_name":
            "gemma-3-4b-it",

        "display_name":
            "Gemma-3-4B-IT",

        "family":
            "gemma3",

        "num_layers":
            34,

        "layers":
            [
                3,
                7,
                10,
                14,
                17,
                20,
                24,
                27,
                31,
                34,
            ],
    },


    "qwen-4b": {

        "model_id":
            "Qwen/Qwen3-4B",

        "save_name":
            "qwen3-4b",

        "display_name":
            "Qwen3-4B",

        "family":
            "qwen3",

        "num_layers":
            36,

        "layers":
            [
                4,
                7,
                11,
                14,
                18,
                22,
                25,
                29,
                32,
                36,
            ],
    },


    "llama-8b": {

        "model_id":
            "meta-llama/Llama-3.1-8B-Instruct",

        "save_name":
            "llama-3.1-8b-instruct",

        "display_name":
            "Llama-3.1-8B-Instruct",

        "family":
            "llama",

        "num_layers":
            32,

        "layers":
            [
                3,
                6,
                10,
                13,
                16,
                19,
                22,
                26,
                29,
                32,
            ],
    },


    "gemma-12b": {

        "model_id":
            "google/gemma-3-12b-it",

        "save_name":
            "gemma-3-12b-it",

        "display_name":
            "Gemma-3-12B-IT",

        "family":
            "gemma3",

        "num_layers":
            48,

        "layers":
            [
                5,
                10,
                14,
                19,
                24,
                29,
                34,
                38,
                43,
                48,
            ],
    },


    "qwen-14b": {

        "model_id":
            "Qwen/Qwen3-14B",

        "save_name":
            "qwen3-14b",

        "display_name":
            "Qwen3-14B",

        "family":
            "qwen3",

        "num_layers":
            40,

        "layers":
            [
                4,
                8,
                12,
                16,
                20,
                24,
                28,
                32,
                36,
                40,
            ],
    },
}


if MODEL_KEY not in MODEL_CONFIGS:

    raise ValueError(
        f"Unknown MODEL_KEY: {MODEL_KEY}"
    )


CFG = MODEL_CONFIGS[
    MODEL_KEY
]


MODEL_ID = CFG[
    "model_id"
]


MODEL_SAVE_NAME = CFG[
    "save_name"
]


MODEL_DISPLAY_NAME = CFG[
    "display_name"
]


MODEL_FAMILY = CFG[
    "family"
]


NUM_LAYERS_EXPECTED = CFG[
    "num_layers"
]


INTERVENTION_LAYERS = CFG[
    "layers"
]



# 5. VALUES / OBJECTIVES


FOUNDATIONS = [

    "Care",

    "Fairness",

    "Loyalty",

    "Authority",

    "Sanctity",
]


FOUNDATION_TO_INDEX = {

    value:
        i

    for i, value
    in enumerate(
        FOUNDATIONS
    )
}


OBJECTIVES = {

    "care_fairness": {

        "Care":
            +1,

        "Fairness":
            +1,
    },


    "loyalty_authority": {

        "Loyalty":
            +1,

        "Authority":
            +1,
    },


    "care_fairness_sanctity": {

        "Care":
            +1,

        "Fairness":
            +1,

        "Sanctity":
            -1,
    },
}



# 6. GENERATION CONFIG


GAMMA = 1.0

MAX_NEW_TOKENS = 128

DO_SAMPLE = False

REPETITION_PENALTY = 1.0

GENERATION_SEED = 1234


SAVE_EVERY_N = 25

RESUME = True



# 7. HF PATHS


HF_PROMPT_MANIFEST = (

    "artifacts/multivalue_steering/"
    f"manifests/{VERSION}/"
    "multivalue_prompt_manifest.csv"
)


HF_OBJECTIVES = (

    "artifacts/multivalue_steering/"
    f"manifests/{VERSION}/"
    "multivalue_objectives.json"
)


HF_DIRECTION_FILE = (

    f"artifacts/value_directions/"
    f"{MODEL_SAVE_NAME}/{VERSION}/directions/"
    f"{MODEL_SAVE_NAME}_"
    f"mft_value_directions_{VERSION}.safetensors"
)


HF_GENERATION_ROOT = (

    f"artifacts/multivalue_steering/"
    f"{MODEL_SAVE_NAME}/{VERSION}/generations/"
    f"{METHOD}"
)



# 8. DRIVE / LOCAL PATHS


if not Path(
    "/content/drive/MyDrive"
).exists():

    drive.mount(
        "/content/drive"
    )


DRIVE_ROOT = Path(

    "/content/drive/MyDrive/"
    "AIMES/multivalue_steering/"
    f"{MODEL_SAVE_NAME}/{VERSION}/generations/"
    f"{METHOD}"
)


DRIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


LOCAL_ROOT = Path(

    f"/content/"
    f"aimes_multivalue_"
    f"{MODEL_SAVE_NAME}_{METHOD}"
)


LOCAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


CHECKPOINT_CSV = (

    DRIVE_ROOT
    /
    "generations_checkpoint.csv"
)



# 9. HELPERS


def clear_memory():

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


def get_dtype():

    if not torch.cuda.is_available():

        return torch.float32


    if torch.cuda.is_bf16_supported():

        return torch.bfloat16


    return torch.float16


def get_input_device(
    model,
):

    for getter in [

        lambda:
            model
            .get_input_embeddings()
            .weight.device,

        lambda:
            model
            .language_model
            .get_input_embeddings()
            .weight.device,

        lambda:
            model
            .model.language_model
            .get_input_embeddings()
            .weight.device,
    ]:

        try:

            return getter()

        except Exception:

            pass


    raise RuntimeError(
        "Could not determine input device."
    )


def get_transformer_layers(
    model,
):

    for getter in [

        lambda:
            model.model.layers,

        lambda:
            model.model.language_model.layers,

        lambda:
            model.language_model.layers,

        lambda:
            model.language_model.model.layers,
    ]:

        try:

            layers = getter()


            if len(
                layers
            ) == NUM_LAYERS_EXPECTED:

                return layers


        except Exception:

            pass


    raise RuntimeError(
        "Could not locate transformer layers."
    )


def seed_all(
    seed,
):

    random.seed(
        seed
    )


    np.random.seed(
        seed
    )


    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


def write_json(
    path,
    obj,
):

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False,
        )


def condition_id(
    prompt_id,
    objective_id,
    layer,
):

    payload = "|".join(

        [

            MODEL_SAVE_NAME,

            str(
                prompt_id
            ),

            objective_id,

            str(
                layer
            ),

            METHOD,

            str(
                GAMMA
            ),
        ]
    )


    return hashlib.sha256(

        payload.encode(
            "utf-8"
        )

    ).hexdigest()



# 10. LOAD SHARED PROMPT MANIFEST
#
# This must already exist because LL/JL use the exact same
# prompt set.


if not hf_api.file_exists(

    repo_id=
        HF_REPO_ID,

    filename=
        HF_PROMPT_MANIFEST,

    repo_type=
        "model",

    token=
        HF_TOKEN,
):

    raise RuntimeError(

        "Shared multi-value prompt manifest is missing:\n"
        f"{HF_PROMPT_MANIFEST}"
    )


manifest_local = hf_hub_download(

    repo_id=
        HF_REPO_ID,

    filename=
        HF_PROMPT_MANIFEST,

    repo_type=
        "model",

    token=
        HF_TOKEN,
)


manifest_df = pd.read_csv(
    manifest_local
)


required_manifest_columns = {

    "prompt_id",

    "foundation",

    "prompt",
}


missing = (

    required_manifest_columns

    -

    set(
        manifest_df.columns
    )
)


if missing:

    raise RuntimeError(

        f"Manifest missing columns: "
        f"{sorted(missing)}"
    )


if len(
    manifest_df
) != 100:

    raise RuntimeError(

        f"Expected 100 prompts, "
        f"found {len(manifest_df)}."
    )


print(
    "\nShared manifest loaded:"
)


print(
    len(
        manifest_df
    ),
    "prompts"
)



# 11. VALIDATE STORED OBJECTIVES


if not hf_api.file_exists(

    repo_id=
        HF_REPO_ID,

    filename=
        HF_OBJECTIVES,

    repo_type=
        "model",

    token=
        HF_TOKEN,
):

    raise RuntimeError(

        "Shared objective artifact missing:\n"
        f"{HF_OBJECTIVES}"
    )


objective_local = hf_hub_download(

    repo_id=
        HF_REPO_ID,

    filename=
        HF_OBJECTIVES,

    repo_type=
        "model",

    token=
        HF_TOKEN,
)


with open(
    objective_local,
    "r",
    encoding="utf-8",
) as f:

    stored_objectives = json.load(
        f
    )


if stored_objectives != OBJECTIVES:

    raise RuntimeError(

        "Stored objective artifact does not match "
        "this script."
    )



# 12. LOAD MODEL


dtype = get_dtype()


print(
    "\nLoading model:"
)


print(
    MODEL_DISPLAY_NAME
)


print(
    "dtype:",
    dtype
)


if MODEL_FAMILY == "gemma3":

    processor = AutoProcessor.from_pretrained(

        MODEL_ID,

        token=
            HF_TOKEN,
    )


    tokenizer = processor.tokenizer


    model = (

        Gemma3ForConditionalGeneration

        .from_pretrained(

            MODEL_ID,

            token=
                HF_TOKEN,

            dtype=
                dtype,

            device_map=
                "auto",

            low_cpu_mem_usage=
                True,
        )

        .eval()
    )


else:

    processor = None


    tokenizer = AutoTokenizer.from_pretrained(

        MODEL_ID,

        token=
            HF_TOKEN,

        trust_remote_code=
            True,
    )


    model = (

        AutoModelForCausalLM

        .from_pretrained(

            MODEL_ID,

            token=
                HF_TOKEN,

            dtype=
                dtype,

            device_map=
                "auto",

            low_cpu_mem_usage=
                True,

            trust_remote_code=
                True,
        )

        .eval()
    )


if tokenizer.pad_token_id is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


INPUT_DEVICE = get_input_device(
    model
)


TRANSFORMER_LAYERS = get_transformer_layers(
    model
)


print(
    "Transformer layers:",
    len(
        TRANSFORMER_LAYERS
    )
)



# 13. LOAD VALUE DIRECTIONS


direction_local = hf_hub_download(

    repo_id=
        HF_REPO_ID,

    filename=
        HF_DIRECTION_FILE,

    repo_type=
        "model",

    token=
        HF_TOKEN,
)


UNIT_DIRECTIONS = (

    load_file(
        direction_local
    )[
        "unit_directions"
    ]

    .float()

    .cpu()
)


if UNIT_DIRECTIONS.shape[
    0
] != len(
    FOUNDATIONS
):

    raise RuntimeError(

        "Direction tensor value dimension mismatch."
    )


if UNIT_DIRECTIONS.shape[
    1
] != NUM_LAYERS_EXPECTED:

    raise RuntimeError(

        "Direction tensor layer dimension mismatch."
    )


print(
    "Direction tensor:",
    tuple(
        UNIT_DIRECTIONS.shape
    )
)



# 14. PROMPT FORMATTING
#
# Same conventions as AIMES-LL/JL.


def render_prompt(
    prompt,
):

    if MODEL_FAMILY == "gemma3":

        return processor.apply_chat_template(

            [

                {
                    "role":
                        "user",

                    "content": [

                        {
                            "type":
                                "text",

                            "text":
                                str(
                                    prompt
                                ),
                        }
                    ],
                }
            ],

            tokenize=False,

            add_generation_prompt=True,
        )


    kwargs = {

        "tokenize":
            False,

        "add_generation_prompt":
            True,
    }


    if MODEL_FAMILY == "qwen3":

        kwargs[
            "enable_thinking"
        ] = False


    messages = [

        {
            "role":
                "user",

            "content":
                str(
                    prompt
                ),
        }
    ]


    try:

        return tokenizer.apply_chat_template(

            messages,

            **kwargs,
        )


    except TypeError:

        kwargs.pop(
            "enable_thinking",
            None,
        )


        return tokenizer.apply_chat_template(

            messages,

            **kwargs,
        )


def encode_prompt(
    prompt,
):

    encoded = tokenizer(

        render_prompt(
            prompt
        ),

        return_tensors=
            "pt",

        add_special_tokens=
            False,
    )


    return {

        key:
            value.to(
                INPUT_DEVICE
            )

        for key, value
        in encoded.items()

        if torch.is_tensor(
            value
        )
    }



# 15. FIXED MULTI CONTROLLER


class FixedMultiController:

    def __init__(
        self,
        objective,
        layer,
    ):

        self.objective = objective

        self.layer = int(
            layer
        )

        self.layer_idx = (

            self.layer

            -

            1
        )


    def __call__(
        self,
        module,
        inputs,
        output,
    ):

        if torch.is_tensor(
            output
        ):

            hidden = output

            output_type = "tensor"


        elif isinstance(
            output,
            tuple,
        ):

            hidden = output[
                0
            ]

            output_type = "tuple"


        else:

            raise RuntimeError(

                "Unexpected transformer layer output."
            )


        modified = hidden.clone()


        h = hidden[
            :,
            -1,
            :
        ]


        steering = torch.zeros_like(
            h
        )


        for value, sign in self.objective.items():

            direction = UNIT_DIRECTIONS[

                FOUNDATION_TO_INDEX[
                    value
                ],

                self.layer_idx,

            ].to(

                h.device,

                dtype=
                    h.dtype,
            )


            steering += (

                float(
                    sign
                )

                *

                direction[
                    None,
                    :
                ]
            )


        modified[
            :,
            -1,
            :
        ] += (

            GAMMA

            *

            steering
        )


        if output_type == "tensor":

            return modified


        return (

            modified,

            *output[
                1:
            ],
        )



# 16. BASELINE GENERATION


@torch.inference_mode()
def baseline_generate(
    prompt,
    seed,
):

    seed_all(
        seed
    )


    inputs = encode_prompt(
        prompt
    )


    prompt_len = int(

        inputs[
            "input_ids"
        ].shape[
            1
        ]
    )


    output = model.generate(

        **inputs,

        max_new_tokens=
            MAX_NEW_TOKENS,

        do_sample=
            DO_SAMPLE,

        repetition_penalty=
            REPETITION_PENALTY,

        pad_token_id=
            tokenizer.pad_token_id,

        eos_token_id=
            tokenizer.eos_token_id,

        use_cache=
            True,
    )


    text = tokenizer.decode(

        output[
            0,
            prompt_len:
        ],

        skip_special_tokens=
            True,

    ).strip()


    del inputs

    del output


    clear_memory()


    return text



# 17. FIXED MULTI GENERATION


@torch.inference_mode()
def steered_generate(
    prompt,
    seed,
    objective,
    layer,
):

    seed_all(
        seed
    )


    inputs = encode_prompt(
        prompt
    )


    prompt_len = int(

        inputs[
            "input_ids"
        ].shape[
            1
        ]
    )


    controller = FixedMultiController(

        objective=
            objective,

        layer=
            layer,
    )


    handle = TRANSFORMER_LAYERS[

        layer
        -
        1

    ].register_forward_hook(
        controller
    )


    try:

        output = model.generate(

            **inputs,

            max_new_tokens=
                MAX_NEW_TOKENS,

            do_sample=
                DO_SAMPLE,

            repetition_penalty=
                REPETITION_PENALTY,

            pad_token_id=
                tokenizer.pad_token_id,

            eos_token_id=
                tokenizer.eos_token_id,

            use_cache=
                True,
        )


    finally:

        handle.remove()


    text = tokenizer.decode(

        output[
            0,
            prompt_len:
        ],

        skip_special_tokens=
            True,

    ).strip()


    del inputs

    del output


    clear_memory()


    return text



# 18. RESUME CHECKPOINT


rows = []


if (
    RESUME

    and

    CHECKPOINT_CSV.exists()
):

    checkpoint_df = pd.read_csv(
        CHECKPOINT_CSV
    )


    if len(
        checkpoint_df
    ) > 0:

        if "method" not in checkpoint_df.columns:

            raise RuntimeError(

                "Existing checkpoint is incompatible."
            )


        observed_methods = set(

            checkpoint_df[
                "method"
            ]

            .astype(
                str
            )

            .unique()
        )


        if observed_methods != {
            METHOD
        }:

            raise RuntimeError(

                "Existing checkpoint belongs to "
                "a different method."
            )


        rows = checkpoint_df.to_dict(
            orient="records"
        )


completed = {

    str(
        row[
            "condition_id"
        ]
    )

    for row
    in rows
}


baseline_cache = {}


for row in rows:

    baseline_cache[

        str(
            row[
                "prompt_id"
            ]
        )

    ] = str(

        row[
            "baseline_response"
        ]
    )


print(
    "\nAlready completed:",
    len(
        rows
    )
)



# 19. EXPECTED SIZE


EXPECTED_ROWS = (

    len(
        manifest_df
    )

    *

    len(
        OBJECTIVES
    )

    *

    len(
        INTERVENTION_LAYERS
    )
)


if EXPECTED_ROWS != 3000:

    raise RuntimeError(

        f"Expected design should produce 3000 rows, "
        f"but calculated {EXPECTED_ROWS}."
    )


print(
    "Expected rows:",
    EXPECTED_ROWS
)



# 20. GENERATION LOOP


new_since_save = 0


for prompt_index, prompt_row in manifest_df.iterrows():

    prompt_id = str(
        prompt_row[
            "prompt_id"
        ]
    )


    foundation = str(
        prompt_row[
            "foundation"
        ]
    )


    prompt = str(
        prompt_row[
            "prompt"
        ]
    )


    seed = (

        GENERATION_SEED

        +

        int(
            prompt_index
        )
    )


    # --------------------------------------------------------
    # Generate baseline exactly once per prompt.
    # --------------------------------------------------------

    if prompt_id not in baseline_cache:

        print(

            f"\nBaseline "
            f"{prompt_index + 1}/"
            f"{len(manifest_df)}"
        )


        baseline_cache[
            prompt_id
        ] = baseline_generate(

            prompt=
                prompt,

            seed=
                seed,
        )


    baseline = baseline_cache[
        prompt_id
    ]


    # --------------------------------------------------------
    # 3 objectives x 10 depths
    # --------------------------------------------------------

    for objective_id, objective in OBJECTIVES.items():

        for layer in INTERVENTION_LAYERS:

            cid = condition_id(

                prompt_id=
                    prompt_id,

                objective_id=
                    objective_id,

                layer=
                    layer,
            )


            if cid in completed:

                continue


            print(

                f"\r"
                f"{MODEL_SAVE_NAME} | "
                f"{METHOD} | "
                f"{prompt_index + 1:03d}/"
                f"{len(manifest_df)} | "
                f"{objective_id:28s} | "
                f"L{layer:02d} | "
                f"{len(rows)+1:04d}/"
                f"{EXPECTED_ROWS}",

                end="",
            )


            response = steered_generate(

                prompt=
                    prompt,

                seed=
                    seed,

                objective=
                    objective,

                layer=
                    layer,
            )


            row = {

                # =============================================
                # Identity
                # =============================================

                "condition_id":
                    cid,

                "model_key":
                    MODEL_KEY,

                "model_id":
                    MODEL_ID,

                "model_save_name":
                    MODEL_SAVE_NAME,

                "version":
                    VERSION,

                "method":
                    METHOD,


                # =============================================
                # Prompt
                # =============================================

                "prompt_id":
                    prompt_id,

                "prompt_foundation":
                    foundation,

                "prompt":
                    prompt,


                # =============================================
                # Objective
                # =============================================

                "objective_id":
                    objective_id,

                "requested_values":
                    json.dumps(

                        list(
                            objective.keys()
                        )
                    ),

                "requested_signs":
                    json.dumps(

                        list(
                            objective.values()
                        )
                    ),

                "objective_json":
                    json.dumps(

                        objective,

                        sort_keys=True,
                    ),


                # =============================================
                # Layer
                # =============================================

                "layer":
                    int(
                        layer
                    ),

                "num_layers":
                    int(
                        NUM_LAYERS_EXPECTED
                    ),

                "relative_depth":
                    float(

                        layer

                        /

                        NUM_LAYERS_EXPECTED
                    ),

                "gamma":
                    float(
                        GAMMA
                    ),


                # =============================================
                # Responses
                # =============================================

                "baseline_response":
                    baseline,

                "steered_response":
                    response,

                "response_changed":
                    int(

                        response

                        !=

                        baseline
                    ),


                # =============================================
                # Generation
                # =============================================

                "generation_seed":
                    int(
                        seed
                    ),

                "max_new_tokens":
                    int(
                        MAX_NEW_TOKENS
                    ),

                "do_sample":
                    bool(
                        DO_SAMPLE
                    ),

                "repetition_penalty":
                    float(
                        REPETITION_PENALTY
                    ),


                # =============================================
                # Audit
                # =============================================

                "created_at":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
            }


            rows.append(
                row
            )


            completed.add(
                cid
            )


            new_since_save += 1


            if (
                new_since_save
                >=
                SAVE_EVERY_N
            ):

                pd.DataFrame(
                    rows
                ).to_csv(

                    CHECKPOINT_CSV,

                    index=False,
                )


                print(

                    f"\nCheckpoint: "
                    f"{len(rows)}/"
                    f"{EXPECTED_ROWS}"
                )


                new_since_save = 0


print()



# 21. FINAL VALIDATION


df = pd.DataFrame(
    rows
)


if len(
    df
) != EXPECTED_ROWS:

    raise RuntimeError(

        f"Incomplete Fixed Multi generation.\n"
        f"Expected {EXPECTED_ROWS}, "
        f"found {len(df)}."
    )


if df[
    "condition_id"
].duplicated().any():

    raise RuntimeError(

        "Duplicate condition IDs."
    )


if set(

    df[
        "method"
    ]

    .astype(
        str
    )

    .unique()

) != {

    METHOD

}:

    raise RuntimeError(

        "Unexpected method values."
    )


if df[
    "prompt_id"
].nunique() != 100:

    raise RuntimeError(

        "Expected 100 unique prompts."
    )


if df[
    "objective_id"
].nunique() != 3:

    raise RuntimeError(

        "Expected 3 objectives."
    )


if df[
    "relative_depth"
].nunique() != 10:

    raise RuntimeError(

        "Expected 10 intervention depths."
    )


# ------------------------------------------------------------
# Every prompt should have:
#
# 3 objectives x 10 layers = 30 conditions.
# ------------------------------------------------------------

conditions_per_prompt = (

    df

    .groupby(
        "prompt_id"
    )

    .size()
)


if not (

    conditions_per_prompt

    ==
    30

).all():

    raise RuntimeError(

        "Each prompt should have exactly 30 "
        "Fixed Multi conditions."
    )


print(
    "\n"
    +
    "=" * 100
)


print(
    "VALIDATION PASSED"
)


print(
    "=" * 100
)


print(
    "Model:",
    MODEL_DISPLAY_NAME
)


print(
    "Method:",
    METHOD
)


print(
    "Rows:",
    len(
        df
    )
)


print(
    "Prompts:",
    df[
        "prompt_id"
    ].nunique()
)


print(
    "Objectives:",
    df[
        "objective_id"
    ].nunique()
)


print(
    "Depths:",
    df[
        "relative_depth"
    ].nunique()
)



# 22. SAVE FINAL FILES


GENERATION_CSV = (

    LOCAL_ROOT
    /
    "multivalue_generations.csv"
)


GENERATION_JSONL = (

    LOCAL_ROOT
    /
    "multivalue_generations.jsonl"
)


METADATA_JSON = (

    LOCAL_ROOT
    /
    "generation_metadata.json"
)


df.to_csv(

    GENERATION_CSV,

    index=False,
)


df.to_json(

    GENERATION_JSONL,

    orient=
        "records",

    lines=
        True,

    force_ascii=
        False,
)


metadata = {

    "experiment":
        (
            "AIMES Fixed Multi-Value Steering"
        ),

    "method":
        METHOD,

    "model_key":
        MODEL_KEY,

    "model_id":
        MODEL_ID,

    "model_save_name":
        MODEL_SAVE_NAME,

    "model_display_name":
        MODEL_DISPLAY_NAME,

    "objectives":
        OBJECTIVES,

    "layers":
        INTERVENTION_LAYERS,

    "gamma":
        GAMMA,

    "n_prompts":
        int(
            len(
                manifest_df
            )
        ),

    "n_objectives":
        int(
            len(
                OBJECTIVES
            )
        ),

    "n_depths":
        int(
            len(
                INTERVENTION_LAYERS
            )
        ),

    "expected_rows":
        EXPECTED_ROWS,

    "generation_seed":
        GENERATION_SEED,

    "max_new_tokens":
        MAX_NEW_TOKENS,

    "do_sample":
        DO_SAMPLE,

    "repetition_penalty":
        REPETITION_PENALTY,

    "controller":
        (
            "h' = h + gamma * "
            "sum_k g_k u_{k,l}"
        ),

    "observer":
        None,

    "adaptive":
        False,

    "created_at":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


write_json(

    METADATA_JSON,

    metadata,
)



# 23. ONE FINAL HF COMMIT


upload_files = [

    GENERATION_CSV,

    GENERATION_JSONL,

    METADATA_JSON,
]


operations = [

    CommitOperationAdd(

        path_in_repo=
            (
                f"{HF_GENERATION_ROOT}/"
                f"{Path(path).name}"
            ),

        path_or_fileobj=
            str(
                path
            ),
    )

    for path
    in upload_files
]


print(
    "\nUploading Fixed Multi artifacts..."
)


commit = hf_api.create_commit(

    repo_id=
        HF_REPO_ID,

    repo_type=
        "model",

    operations=
        operations,

    commit_message=
        (
            "Add Fixed Multi generations "
            f"for {MODEL_SAVE_NAME}"
        ),

    token=
        HF_TOKEN,
)



# 24. DONE


print(
    "\n"
    +
    "=" * 100
)


print(
    "FIXED MULTI GENERATION COMPLETE"
)


print(
    "=" * 100
)


print(
    "Model:",
    MODEL_DISPLAY_NAME
)


print(
    "Rows:",
    len(
        df
    )
)


print(
    "\nHF:"
)


print(
    HF_GENERATION_ROOT
)


print(
    "\nCommit:"
)


print(
    commit
)